# Color Scales and Correlation in Python using the Iris Dataset

This notebook converts the original R Markdown examples into **Python only** and uses the **Iris dataset** instead of `mtcars`.

Libraries used:
- `pandas`
- `matplotlib`
- `seaborn`
- `scipy`
- `sklearn`


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from scipy import stats

sns.set_theme(style="whitegrid")

# Load iris dataset
iris_raw = load_iris(as_frame=True)
df = iris_raw.frame.copy()

# Rename columns for cleaner labels
df.columns = [c.replace(' (cm)', '').replace(' ', '_') for c in df.columns]
df["species"] = df["target"].map(dict(enumerate(iris_raw.target_names)))

df.head()


## 1) Sequential color scale (continuous)

In [ ]:

plt.figure(figsize=(7,5))
plt.scatter(
    df["sepal_length"], 
    df["petal_length"],
    c=df["sepal_width"],
    cmap="magma",
    s=60
)
plt.colorbar(label="sepal_width")
plt.xlabel("sepal_length")
plt.ylabel("petal_length")
plt.title("Sequential (continuous): magma")
plt.show()


In [ ]:

plt.figure(figsize=(7,5))
plt.scatter(
    df["sepal_length"], 
    df["petal_length"],
    c=df["sepal_width"],
    cmap="Blues",
    s=60
)
plt.colorbar(label="sepal_width")
plt.xlabel("sepal_length")
plt.ylabel("petal_length")
plt.title("Sequential (continuous): Blues")
plt.show()


## 2) Diverging color scale (continuous)

In [ ]:

df["sepal_width_centered"] = df["sepal_width"] - df["sepal_width"].mean()

plt.figure(figsize=(7,5))
plt.scatter(
    df["sepal_length"], 
    df["petal_length"],
    c=df["sepal_width_centered"],
    cmap="RdBu",
    s=60
)
plt.colorbar(label="sepal_width - mean(sepal_width)")
plt.xlabel("sepal_length")
plt.ylabel("petal_length")
plt.title("Diverging (continuous): RdBu")
plt.show()


## 3) Qualitative color scales (discrete categories)

In [ ]:

species_counts = df["species"].value_counts().reset_index()
species_counts.columns = ["species", "count"]
species_counts


In [ ]:

plt.figure(figsize=(7,5))
sns.barplot(data=species_counts, x="species", y="count", palette="Dark2", hue="species", dodge=False, legend=False)
plt.title("Qualitative (discrete): Dark2")
plt.xlabel("")
plt.ylabel("count")
plt.xticks(rotation=20)
plt.show()


In [ ]:

plt.figure(figsize=(7,5))
sns.barplot(data=species_counts, x="species", y="count", hue="species", dodge=False, legend=False)
plt.title("Qualitative: seaborn default palette")
plt.xlabel("")
plt.ylabel("count")
plt.xticks(rotation=20)
plt.show()


## 4) Manual coloring

In [ ]:

manual_colors = ["#1b9e77", "#d95f02", "#7570b3"]

plt.figure(figsize=(7,5))
sns.barplot(data=species_counts, x="species", y="count", palette=manual_colors, hue="species", dodge=False, legend=False)
plt.title("Qualitative: manual colors")
plt.xlabel("")
plt.ylabel("count")
plt.xticks(rotation=20)
plt.show()


## 5) Gray scale

In [ ]:

grey_colors = ["#4d4d4d", "#969696", "#d9d9d9"]

plt.figure(figsize=(7,5))
sns.barplot(data=species_counts, x="species", y="count", palette=grey_colors, hue="species", dodge=False, legend=False)
plt.title("Qualitative: grey scale")
plt.xlabel("")
plt.ylabel("count")
plt.xticks(rotation=20)
plt.show()


## What each type shows

- **Sequential:** for ordered numeric values.
- **Diverging:** for values centered around an important reference, such as the mean or zero.
- **Qualitative:** for categories such as species.
- **Manual:** when you want specific custom colors.
- **Grey:** useful for printing in black and white.


## Correlation coefficient summary

Before calculating correlation, it is common to inspect:
- histograms
- Q-Q plots
- scatter plots
- normality tests such as **Shapiro-Wilk**


In [ ]:

numeric_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
iris_num = df[numeric_cols].copy()
iris_num.head()


### Histograms

In [ ]:

iris_num.hist(figsize=(10,8), edgecolor="black")
plt.suptitle("Histograms of Iris numeric variables", y=1.02)
plt.tight_layout()
plt.show()


### Q-Q plots

In [ ]:

for col in numeric_cols:
    plt.figure(figsize=(5,4))
    stats.probplot(iris_num[col], dist="norm", plot=plt)
    plt.title(f"Q-Q Plot: {col}")
    plt.show()


### Shapiro-Wilk normality tests

In [ ]:

shapiro_results = []

for col in numeric_cols:
    stat, p_value = stats.shapiro(iris_num[col])
    shapiro_results.append({
        "variable": col,
        "W_statistic": stat,
        "p_value": p_value
    })

shapiro_df = pd.DataFrame(shapiro_results)
shapiro_df


### Scatter plot matrix

In [ ]:

sns.pairplot(df[numeric_cols + ["species"]], hue="species")
plt.show()


### Correlation matrix

In [ ]:

corr_matrix = iris_num.corr(numeric_only=True)
corr_matrix.round(2)


In [ ]:

plt.figure(figsize=(7,5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()


### Correlation matrix with p-values

In [ ]:

corr = iris_num.corr()
pvals = pd.DataFrame(np.ones((len(numeric_cols), len(numeric_cols))), columns=numeric_cols, index=numeric_cols)

for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        r, p = stats.pearsonr(iris_num[numeric_cols[i]], iris_num[numeric_cols[j]])
        pvals.iloc[i, j] = p

print("Correlation coefficients:")
display(corr.round(3))

print("P-values:")
display(pvals.round(6))


## Notes

- In the original R file, `mtcars` and `mpg` were used.
- Here, everything is rewritten in Python and based on the **Iris** dataset.
- The examples keep the same ideas: color scales, categorical coloring, normality checking, and correlation analysis.
